In [1]:
!pip install pandas_datareader

Looking in indexes: http://nexus.charisma:5080/repository/pypi-proxy/simple


In [26]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from pandas_datareader import data as pdr

START = "1995-01-01"
END   = "2025-12-01"
MARKET = "SPY"  # or "^GSPC"

FRED_SERIES = {
    "DFF": "fed_funds",
    "DGS10": "rate_10y",
    "DGS2": "rate_2y",
    "T10Y2Y": "yc_slope_10y2y",
    "CPIAUCSL": "cpi",
    "UNRATE": "unemp",
    "INDPRO": "indpro",
}

# Evaluation eras (held out from training)
SPLITS = {
    "gfc_2008_2010": ("2008-01-01", "2010-12-31"),
    "covid_2020_2021": ("2020-01-01", "2021-12-31"),
    "inflation_2022_2023": ("2022-01-01", "2023-12-31"),
}

OUT_ROOT = "dataset_finance"


def ensure_dirs():
    os.makedirs(os.path.join(OUT_ROOT, "metadata"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "finetune"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "eval"), exist_ok=True)


def get_sp500_tickers() -> list[str]:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    tickers = pd.read_html(response.text)[0]["Symbol"].astype(str).tolist()
    return [t.replace(".", "-") for t in tickers]


def download_close(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    px = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    price_field = "Close"  # or "Adj Close"

    if isinstance(px.columns, pd.MultiIndex):
        close = pd.concat(
            {t: px[t][price_field] for t in tickers if t in px.columns.get_level_values(0)},
            axis=1,
        )
    else:
        close = px[price_field].to_frame(tickers[0])

    return close.sort_index()


def log_returns(close_df: pd.DataFrame) -> pd.DataFrame:
    return np.log(close_df).diff()


def download_fred(series_map: dict, start: str, end: str) -> pd.DataFrame:
    macro = pdr.DataReader(list(series_map.keys()), "fred", start, end)
    return macro.rename(columns=series_map).sort_index()


def save_metadata(tickers: list[str]):
    pd.Series(tickers, name="ticker").to_csv(os.path.join(OUT_ROOT, "metadata", "tickers.csv"), index=False)
    pd.DataFrame({"fred_code": list(FRED_SERIES.keys()), "name": list(FRED_SERIES.values())}).to_csv(
        os.path.join(OUT_ROOT, "metadata", "fred_series.csv"), index=False
    )
    with open(os.path.join(OUT_ROOT, "metadata", "build_config.json"), "w") as f:
        json.dump(
            {"start": START, "end": END, "market": MARKET, "fred_series": FRED_SERIES, "splits": SPLITS},
            f,
            indent=2,
        )


def _build_exclusion_mask(index: pd.DatetimeIndex, splits: dict) -> pd.Series:
    """True = keep for training, False = exclude (eval eras)."""
    keep = pd.Series(True, index=index)
    for _, (a, b) in splits.items():
        keep.loc[a:b] = False
    return keep


def _contiguous_segments_from_masked_series(
    s: pd.Series,
    keep_mask: pd.Series,
    min_len: int,
    max_gap_days: int = 7,
):
    """
    Returns list of contiguous sequences from s where keep_mask is True.
    Contiguity is defined by date gaps <= max_gap_days (trading calendar gaps OK).
    """
    # Keep only dates allowed for training
    s2 = s[keep_mask].dropna()
    if len(s2) == 0:
        return []

    # Identify breaks where the date gap is too large
    dt = s2.index.to_series().diff().dt.days.fillna(1)
    # A large gap implies we're jumping across excluded eras or missing chunks
    breaks = (dt > max_gap_days).cumsum()

    segments = []
    for _, seg in s2.groupby(breaks):
        if len(seg) >= min_len:
            segments.append(seg.astype(float).to_list())
    return segments


def write_train_jsonl_contiguous(
    returns_df: pd.DataFrame,
    splits: dict,
    out_path: str,
    min_len: int = 512,
    max_gap_days: int = 7,
):
    """
    Writes ONE jsonl file with MANY sequences (recommended for your training pipeline).
    Each line:
      {"ticker":"AAPL","segment_id":0,"sequence":[...]}
    """
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    keep_mask = _build_exclusion_mask(returns_df.index, splits)

    n_seq = 0
    n_tickers = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for t in returns_df.columns:
            s = returns_df[t]
            segs = _contiguous_segments_from_masked_series(
                s, keep_mask=keep_mask, min_len=min_len, max_gap_days=max_gap_days
            )
            if not segs:
                continue

            n_tickers += 1
            for j, seq in enumerate(segs):
                obj = {"ticker": t, "segment_id": j, "sequence": seq}
                f.write(json.dumps(obj) + "\n")
                n_seq += 1

    total_dates = len(returns_df.index)
    train_dates = int(keep_mask.sum())
    eval_dates = total_dates - train_dates

    print(f"[split] total dates={total_dates} | train dates={train_dates} | eval-excluded dates={eval_dates}")
    print(f"[train.jsonl] wrote {n_seq} sequences from {n_tickers} tickers to {out_path}")


def write_eval_csvs(panel_wide: pd.DataFrame, splits: dict):
    eval_dir = os.path.join(OUT_ROOT, "eval")
    os.makedirs(eval_dir, exist_ok=True)

    for name, (a, b) in splits.items():
        df = panel_wide.loc[a:b].copy()
        df.reset_index().rename(columns={"index": "date"}).to_csv(os.path.join(eval_dir, f"{name}.csv"), index=False)


def main():
    ensure_dirs()

    tickers = get_sp500_tickers()
    all_tickers = sorted(set(tickers + [MARKET]))
    save_metadata(tickers)

    # 1) prices -> returns
    close = download_close(all_tickers, START, END)
    rets = log_returns(close)

    market_ret = rets[MARKET].rename("market_ret_1d")

    # 2) macro aligned, ffill, shift(1)
    macro = download_fred(FRED_SERIES, START, END)
    idx = rets.index
    macro = macro.reindex(idx).ffill().shift(1)

    # 3) panel for eval csvs
    firm_rets = rets.drop(columns=[MARKET], errors="ignore")
    panel_wide = firm_rets.copy()
    panel_wide["market_ret_1d"] = market_ret
    for c in macro.columns:
        panel_wide[c] = macro[c]
    panel_wide = panel_wide.dropna(subset=["market_ret_1d"])

    # 4) Write training jsonl: contiguous segments, no fake jumps
    out_train = os.path.join(OUT_ROOT, "finetune", "train.jsonl")
    write_train_jsonl_contiguous(
        returns_df=firm_rets,
        splits=SPLITS,
        out_path=out_train,
        min_len=512,         # choose based on your max_length/stride later
        max_gap_days=7,      # treat >7 days as a break
    )

    # Optional: write eval csvs for your run_eval script
    write_eval_csvs(panel_wide, SPLITS)

    print("Done.")
    print(f"Train JSONL: {out_train}")
    print(f"Eval CSVs:   {os.path.join(OUT_ROOT, 'eval')}")


if __name__ == "__main__":
    main()


/tmp/ipykernel_121576/355105004.py:50: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tickers = pd.read_html(response.text)[0]["Symbol"].astype(str).tolist()

17 Failed downloads:
['RVTY']: Timeout('Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 73315 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['WMT']: Timeout('Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 111525 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['NKE']: Timeout('Failed to perform, curl: (28) Operation timed out after 10001 milliseconds with 66546 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['NFLX']: Timeout('Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 155865 

[split] total dates=7780 | train dates=6017 | eval-excluded dates=1763
[train.jsonl] wrote 838 sequences from 459 tickers to dataset_finance/finetune/train.jsonl
Done.
Train JSONL: dataset_finance/finetune/train.jsonl
Eval CSVs:   dataset_finance/eval


In [27]:
import os
import json
from pathlib import Path

# Paths
JSONL_DIR = "dataset_finance/finetune/jsonl"
OUTPUT_FILE = "dataset_finance/finetune/train_sequences.jsonl"
OUTPUT_FILE_SMALL = "dataset_finance/finetune/train_sequences_small.jsonl"


def combine_jsonl_files():
    """
    Combine all JSONL files from jsonl folder into one file.
    Also creates a smaller version with first 10 sequences.
    """
    # Get all JSONL files
    jsonl_path = Path(JSONL_DIR)
    jsonl_files = sorted(jsonl_path.glob("*.jsonl"))
    
    if not jsonl_files:
        print(f"No JSONL files found in {JSONL_DIR}")
        return
    
    print(f"Found {len(jsonl_files)} JSONL files")
    
    # Read all sequences
    all_sequences = []
    total_observations = 0
    
    for jsonl_file in jsonl_files:
        try:
            with open(jsonl_file, 'r') as f:
                data = json.load(f)
                sequence = data.get("sequence", [])
                if sequence:  # Only add non-empty sequences
                    all_sequences.append({"sequence": sequence})
                    total_observations += len(sequence)
        except Exception as e:
            print(f"Warning: Could not read {jsonl_file}: {e}")
            continue
    
    # Write full file
    with open(OUTPUT_FILE, 'w') as f:
        for seq_data in all_sequences:
            f.write(json.dumps(seq_data) + '\n')
    
    # Write small file (first 10 sequences)
    with open(OUTPUT_FILE_SMALL, 'w') as f:
        for seq_data in all_sequences[:10]:
            f.write(json.dumps(seq_data) + '\n')
    
    # Calculate observations for small file
    small_observations = sum(len(seq["sequence"]) for seq in all_sequences[:10])
    
    # Print summary
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Full file ({OUTPUT_FILE}):")
    print(f"  - Sequences: {len(all_sequences):,}")
    print(f"  - Total observations: {total_observations:,}")
    print(f"\nSmall file ({OUTPUT_FILE_SMALL}):")
    print(f"  - Sequences: {min(10, len(all_sequences)):,}")
    print(f"  - Total observations: {small_observations:,}")
    print("="*60)


# Run the function
combine_jsonl_files()


Found 477 JSONL files

SUMMARY
Full file (dataset_finance/finetune/train_sequences.jsonl):
  - Sequences: 477
  - Total observations: 2,470,559

Small file (dataset_finance/finetune/train_sequences_small.jsonl):
  - Sequences: 10
  - Total observations: 53,323


In [15]:
import pandas as pd

file_name = "dataset_finance/eval/full_panel_preview.csv"

df = pd.read_csv(file_name)
# df.dropna(axis=1).to_csv(file_name, index=False)

In [4]:
import numpy as np


npz_path = 'results/eval_preds_attn_small.npz'
npz_path2 = 'results/eval_preds_mamba_small.npz'
npz = np.load(npz_path)
npz2 = np.load(npz_path2)
preds = npz['predictions']
pred2=npz2['predictions']
labels = npz['labels']

mse = float(np.mean((preds - pred2) ** 2))
mae = float(np.mean(np.abs(preds - pred2)))

print('MSE =', mse)
print('MAE =', mae)


MSE = 0.0
MAE = 0.0


In [3]:

npz_path = 'results/eval_preds_mamba_small.npz'
npz = np.load(npz_path)
preds = npz['predictions']
labels = npz['labels']

mse = float(np.mean((preds - labels) ** 2))
mae = float(np.mean(np.abs(preds - labels)))

print('MSE =', mse)
print('MAE =', mae)

MSE = 1.9635088443756104
MAE = 0.8803147077560425


In [1]:
import os, glob, json, pathlib

CKPT_DIR = "logs/time_moe_attn_small"  # <-- change this

paths = sorted(glob.glob(os.path.join(CKPT_DIR, "*")))
print("Checkpoint dir:", CKPT_DIR)
print("Files:")
for p in paths:
    print(" -", os.path.basename(p))


Checkpoint dir: logs/time_moe_attn_small
Files:
 - config.json
 - generation_config.json
 - model.safetensors
 - training_args.bin


In [3]:
import os, glob
from safetensors.torch import safe_open


p = "logs/time_moe_attn_small/model.safetensors"
with safe_open(p, framework="pt", device="cpu") as f:
    for k in f.keys():
        shape = f.get_tensor(k).shape
        n = 1
        for d in shape:
            n *= d
        total_params += n
        tensors += 1

print(f"Found {len(safetensor_paths)} safetensors file(s)")
print(f"Found {tensors} tensors")
print(f"Total parameters: {total_params:,}")


Found 1 safetensors file(s)
Found 657 tensors
Total parameters: 160,458,240


In [4]:
import os

total_bytes = os.path.getsize(p)

print(f"Total checkpoint weight size: {total_bytes/1024**2:.2f} MB ({total_bytes/1024**3:.3f} GB)")


Total checkpoint weight size: 432.46 MB (0.422 GB)


In [5]:
import json, os

config_path = os.path.join(CKPT_DIR, "config.json")
if os.path.exists(config_path):
    config = json.load(open(config_path, "r"))
    print("config.json keys:", list(config.keys())[:30], "..." if len(config.keys()) > 30 else "")
    # Print the most interesting fields if present:
    interesting = [
        "model_type", "hidden_size", "num_hidden_layers", "num_attention_heads",
        "intermediate_size", "num_experts", "num_experts_per_tok",
        "max_position_embeddings", "input_size",
        "mamba_d_state", "mamba_d_conv", "mamba_expand", "temporal_mixer"
    ]
    for k in interesting:
        if k in config:
            print(f"{k}: {config[k]}")
else:
    print("No config.json found in", CKPT_DIR)


config.json keys: ['apply_aux_loss', 'architectures', 'attention_dropout', 'auto_map', 'dtype', 'hidden_act', 'hidden_size', 'horizon_lengths', 'initializer_range', 'input_size', 'intermediate_size', 'mamba_d_conv', 'mamba_d_state', 'mamba_expand', 'max_position_embeddings', 'model_type', 'num_attention_heads', 'num_experts', 'num_experts_per_tok', 'num_hidden_layers', 'num_key_value_heads', 'rms_norm_eps', 'rope_theta', 'router_aux_loss_factor', 'temporal_mixer', 'tie_word_embeddings', 'transformers_version', 'use_cache', 'use_dense'] 
model_type: time_moe
hidden_size: 384
num_hidden_layers: 12
num_attention_heads: 12
intermediate_size: 1536
num_experts: 8
num_experts_per_tok: 2
max_position_embeddings: 4096
input_size: 1
mamba_d_state: 16
mamba_d_conv: 4
mamba_expand: 2
temporal_mixer: attn


In [8]:
summary = {}

# params
summary["total_params"] = total_params
summary["weights_size_gb"] = total_bytes/1024**3

# config bits if available
if os.path.exists(config_path):
    for k in ["hidden_size", "num_hidden_layers", "num_attention_heads", "intermediate_size",
              "num_experts", "num_experts_per_tok", "temporal_mixer"]:
        if k in config:
            summary[k] = config[k]


summary


{'total_params': 160458240,
 'weights_size_gb': 0.4223216474056244,
 'hidden_size': 384,
 'num_hidden_layers': 12,
 'num_attention_heads': 12,
 'intermediate_size': 1536,
 'num_experts': 8,
 'num_experts_per_tok': 2,
 'temporal_mixer': 'attn'}